# Lab 1: Connecting two tables, one step at a time

**CS203 Data Science Fundamentals | Uses Week 1 lecture material**  
**Duration:** 2 hours | **Submit:** one executed notebook by 23:59

You do not need to write a full program today. Read each short code cell, predict what it will do, run it, inspect the output, and answer the question below it.

You will use two tables:

- students: one row is one student;
- enrolments: one row is one student registered in one course.

The central skill is simple: before connecting tables, say what one row means in each table.

## Your route through the lab

| Part | Time | What you do |
| --- | ---: | --- |
| 1. Look | 0:00-0:25 | Run cells and identify rows, columns, and keys |
| 2. Predict | 0:25-0:40 | Decide what should happen when two tables connect |
| 3. Connect | 0:40-1:05 | Complete one short merge and check the row count |
| 4. Summarise | 1:05-1:30 | Complete one group summary and explain its denominator |
| 5. Challenge | 1:30-1:50 | Change one group variable and interpret carefully |
| 6. Explain | 1:50-2:00 | Peer check, final explanation, restart and run all |

There are **10 code cells**. Six are fully provided. You complete four tiny code tasks by replacing one blank or one variable name.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

RAW_BASE = "https://raw.githubusercontent.com/Sabokrou/Sabokrou.github.io/main/teaching/data-science-fundamentals/datasets"

def load_table(name):
    local = Path("data") / name
    if local.exists():
        return pd.read_csv(local)
    return pd.read_csv(f"{RAW_BASE}/{name}")

students = load_table("students.csv")
enrolments = load_table("enrolments.csv")

print("students:", students.shape)
print("enrolments:", enrolments.shape)

## Part 1: Look at the first table

Before running the next cell, predict:

- How many rows do you expect in students?
- Can the same student ID appear more than once?
- Which columns describe a student rather than identify a student?

In [ ]:
students.head()

In [ ]:
print("Is student_id unique in students?", students["student_id"].is_unique)
print("Number of students:", len(students))
print("Columns:", list(students.columns))

### Explain after running

1. **What does one row in students represent?** _Write one sentence._
2. **Why is student_id an identifier rather than a useful numerical measurement?** _Write one sentence._
3. **Choose one column and name its measurement type:** nominal category, number, or identifier. _Write one sentence._

In [ ]:
# TODO 1 - Replace the blank with the table that has one row per student.
# Choose exactly one: students   or   enrolments

one_student_table = _____
one_student_table[["student_id", "programme", "study_mode"]].head()

## Part 2: Look at the second table

Run the next cell. Then compare it with students.

**Predict first:** Should student_id still be unique here? Why or why not?

In [ ]:
print(enrolments.head())
print("Is student_id unique in enrolments?", enrolments["student_id"].is_unique)
print("Number of enrolment rows:", len(enrolments))
print("Number of different students:", enrolments["student_id"].nunique())
print("Courses:", sorted(enrolments["course_code"].unique()))

### Explain after running

1. **What does one row in enrolments represent?** _Write one sentence._
2. **Why can student_id repeat in enrolments without being an error?** _Write one sentence._
3. **Which table should be on the left if your final table needs one row per course registration?** _Write one sentence._

## Part 3: Connect the tables

We want one row per **course registration**, plus the student's programme and study mode.

- enrolments stays on the left because it already has one row per registration.
- students supplies extra student information.
- many_to_one means many enrolment rows can match one student row.

**Predict:** The merge should have the same number of rows as enrolments. Why?

In [ ]:
# TODO 2 - Complete the merge by replacing the blank with one_student_table.
# Do not change on, how, or validate.

joined = enrolments.merge(
    _____[["student_id", "programme", "study_mode"]],
    on="student_id",
    how="left",
    validate="many_to_one",
)

joined.head()

In [ ]:
print("Rows before merge:", len(enrolments))
print("Rows after merge: ", len(joined))
assert len(joined) == len(enrolments), "Unexpected row change: stop and ask for help."
assert joined["programme"].notna().all(), "A student did not match: stop and ask for help."

print("Check passed: one row still represents one student-course registration.")

### Explain after connecting

1. **After the merge, what does one row in joined represent?**
2. **Why does the row count stay the same?**
3. **What would it mean if the merge created twice as many rows?**
4. **Why is many_to_one appropriate here?**

> Answer each in one short sentence.

## Part 4: Make one simple summary

The next task calculates a **course-registration pass rate** by study mode.

This denominator is enrolments, not students. A student taking four courses contributes four registration rows.

**Predict:** Is this the same as “the percentage of students who passed”? Explain before you run the code.

In [ ]:
# TODO 3 - Replace the blank with the column that divides rows into full-time and part-time groups.
# Hint: look at one_student_table.head().

pass_rate_by_mode = (
    joined.groupby(_____)["passed"]
    .agg(n="size", pass_rate="mean")
    .assign(pass_rate=lambda x: (100 * x["pass_rate"]).round(1))
    .reset_index()
)
pass_rate_by_mode

In [ ]:
ax = pass_rate_by_mode.plot.bar(
    x="study_mode", y="pass_rate", legend=False,
    title="Course-registration pass rate by study mode"
)
ax.set_ylabel("Pass rate (%)")
ax.set_xlabel("Study mode")
plt.show()

### Explain the output

1. Report the two group sizes and pass rates.
2. State the denominator in plain English.
3. Complete this sentence:

> “In this dataset, the course-registration pass rates differ by study mode. This describes ________. It does not prove that ________.”

4. Give one possible reason, other than study mode itself, that could help explain a difference.

## Part 5: Small challenge

You have already seen the full pattern. Now change **one word** in the code below.

The question is: “How do course-registration pass rates differ by programme?”

- Replace the blank with the correct grouping column.
- Run the cell.
- Make a short chart or use the printed table.
- Explain why the programme with the highest value is not automatically the “best” programme.

In [ ]:
# TODO 4 - Replace the blank with the correct column name.
# Hint: the column describes Biology, Computer Science, Economics, or Psychology.

challenge_table = (
    joined.groupby(_____)["passed"]
    .agg(n="size", pass_rate="mean")
    .assign(pass_rate=lambda x: (100 * x["pass_rate"]).round(1))
    .reset_index()
)
challenge_table

## Final thinking task

A manager says: “The programme with the highest pass rate is better, so we should copy its teaching everywhere.”

Write **120-160 words**. Use your challenge table, but answer carefully:

- What does the table show?
- What does one row represent?
- What important information is missing?
- Why does a difference not prove that teaching caused it?
- What would you check next?

> _Write here._

## Peer check and submission

Ask a nearby student:

1. What does one row in joined represent?
2. What is the denominator of pass_rate_by_mode?
3. What claim is too strong for this data?

- **Reviewer initials:** _replace_
- **One improvement I made:** _replace_

Before submitting: **Runtime -> Restart session and run all**. All outputs must remain visible.

| Criterion | Weight |
| --- | ---: |
| Rows, tables, and merge explanation | 35% |
| Completed code and reproducible notebook | 25% |
| Correct denominator and summary explanation | 20% |
| Final cautious reasoning task | 15% |
| Peer check and complete submission | 5% |